<a href="https://colab.research.google.com/github/abdullah75f/Transformer-Amharic-bot-new-colab/blob/main/Transformer_Amharic_bot_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Core Libraries ---
import re
import os
import math
import torch

# Install required libraries first
!pip install transformers datasets sentencepiece accelerate -q
print("Required libraries installed/updated.")


# For loading and splitting datasets
from sklearn.model_selection import train_test_split
from datasets import Dataset, load_dataset, load_from_disk

# Hugging Face Transformers - useful tools for training language models
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    pipeline,
    DataCollatorForLanguageModeling
)

# For using Google Drive in Colab (to save or load data/models)
from google.colab import drive

print("Libraries imported.")
# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

# Connects your Google Drive so you can access your files
try:
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted.")
except Exception as e:
    print(f"Error mounting Google Drive: {e}")

In [ ]:
# Set up paths
DRIVE_BASE_PATH = "/content/drive/MyDrive/Amharic_Chatbot"
RAW_DATA_FILE = os.path.join(DRIVE_BASE_PATH, "raw-corpus.txt")

# Paths for saving tokenized datasets and trained models
TOKENIZED_TRAIN_PATH = os.path.join(DRIVE_BASE_PATH, "tokenized_train_dataset")
TOKENIZED_VAL_PATH = os.path.join(DRIVE_BASE_PATH, "tokenized_val_dataset")
MODEL_OUTPUT_DIR = os.path.join(DRIVE_BASE_PATH, "models/amharic-gpt-finetuned")
LOGGING_DIR = os.path.join(DRIVE_BASE_PATH, "logs/amharic-gpt-finetuned")
FINAL_MODEL_PATH = os.path.join(MODEL_OUTPUT_DIR, "final-new")

# Name of the pre-trained base model from Hugging Face
BASE_MODEL_NAME = "rasyosef/gpt2-small-amharic"

# Settings for processing and splitting the dataset
TEST_SPLIT_SIZE = 0.1 # 10% of the data will be used for validation
RANDOM_STATE = 42 # Ensures the split is reproducible
MAX_TOKEN_LENGTH = 128 # Max sequence length for tokenizer

# Training configuration
NUM_TRAIN_EPOCHS = 2
LEARNING_RATE = 5e-5
TRAIN_BATCH_SIZE_PER_DEVICE = 1
GRADIENT_ACCUMULATION_STEPS = 4
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 50
SAVE_STRATEGY = "epoch"
SAVE_TOTAL_LIMIT = 2
FP16_TRAINING = torch.cuda.is_available()

# Use a smaller subset of data for quick testing
DEBUG_MAX_TRAIN_SAMPLES = 2000 # Keep demo size as original

# Parameters for generating text responses from the model
GENERATION_MAX_NEW_TOKENS = 70
GENERATION_TEMPERATURE = 0.8
GENERATION_TOP_K = 40
GENERATION_TOP_P = 0.9
GENERATION_REPETITION_PENALTY = 1.2
GENERATION_NO_REPEAT_NGRAM_SIZE = 3

# Settings for initial test generation (can be different from chatbot settings)
TEST_MAX_NEW_TOKENS = 50
TEST_NUM_BEAMS = 5
TEST_NO_REPEAT_NGRAM_SIZE = 2

print("Configuration set.")
# Create output directories if they don't exist to avoid errors later
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGGING_DIR, exist_ok=True)
print(f"Model output directory: {MODEL_OUTPUT_DIR}")
print(f"Logging directory: {LOGGING_DIR}")

In [ ]:
import re
from sklearn.model_selection import train_test_split

def clean_amharic_v2(text):
    """
    Cleans Amharic text by removing English characters, digits, and unnecessary punctuation.
    Keeps essential Amharic punctuation like '።' and '፣'.
    """
    text = re.sub(r'[a-zA-Z0-9]', '', text)
    text = re.sub(r'[፡;:!?“”"\'`()\[\]{}<>«»‹›\.,\?]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

print(f"Loading raw data from: {RAW_DATA_FILE}")
try:
    with open(RAW_DATA_FILE, 'r', encoding='utf-8') as f:
        raw_texts = f.readlines()
    print(f"Successfully read {len(raw_texts)} lines.")

    # Remove empty lines and extra whitespace
    texts = [line.strip() for line in raw_texts if line.strip()]
    print(f"Initial non-empty lines: {len(texts)}")
    if texts:
        print("Sample line (raw):", texts[0])

        # Clean the text using the function above
        cleaned_texts = [clean_amharic_v2(t) for t in texts]
        # Remove lines that became empty after cleaning
        cleaned_texts = [line for line in cleaned_texts if line]
        print(f"Lines after cleaning: {len(cleaned_texts)}")
        if cleaned_texts:
            print("Sample line (cleaned):", cleaned_texts[0])

        # Split data into training and validation sets
        train_texts, val_texts = train_test_split(
            cleaned_texts,
            test_size=TEST_SPLIT_SIZE,
            random_state=RANDOM_STATE
        )
        print(f"\nSplit complete:")
        print(f"Train samples: {len(train_texts)}")
        print(f"Validation samples: {len(val_texts)}")

        # Save temporary files for later loading into Hugging Face datasets
        TRAIN_TEXT_FILE = 'train.txt'
        VAL_TEXT_FILE = 'val.txt'
        with open(TRAIN_TEXT_FILE, 'w', encoding='utf-8') as f:
            f.write('\n'.join(train_texts))
        with open(VAL_TEXT_FILE, 'w', encoding='utf-8') as f:
            f.write('\n'.join(val_texts))
        print(f"Temporary train/val files saved: {TRAIN_TEXT_FILE}, {VAL_TEXT_FILE}")

    else:
        print("No non-empty lines found in the raw file.")
        train_texts, val_texts = [], []

except FileNotFoundError:
    print(f"Error: File not found at {RAW_DATA_FILE}.")
    train_texts, val_texts = [], []
except Exception as e:
    print(f"An error occurred during data loading/processing: {e}")
    train_texts, val_texts = [], []

In [ ]:
from datasets import load_dataset  # Needed to load raw text into Hugging Face Dataset format
from transformers import AutoTokenizer

# Check if there’s any data to process before proceeding
if not train_texts or not val_texts:
    print("Skipping tokenization as train/validation texts are empty.")
else:
    print(f"\nLoading tokenizer: {BASE_MODEL_NAME}")
    try:
        # Load the tokenizer that matches your base model
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

        # GPT-2 models don't have a padding token by default, so we set it to eos_token
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            print("Set tokenizer pad_token = eos_token")

        # Function to tokenize the text data
        def tokenize_function(examples):
            # Tokenize text, pad/truncate to MAX_TOKEN_LENGTH
            return tokenizer(
                examples['text'],
                padding="max_length", # Pad all sequences to the same length
                truncation=True,      # Cut off any tokens beyond max length
                max_length=MAX_TOKEN_LENGTH
            )

        # Load text data into Hugging Face Dataset format
        print("Loading text files into Dataset objects...")
        train_dataset = load_dataset('text', data_files={'train': TRAIN_TEXT_FILE})['train']
        val_dataset = load_dataset('text', data_files={'validation': VAL_TEXT_FILE})['validation']
        print(f"Loaded {len(train_dataset)} train and {len(val_dataset)} validation examples.")

        # Apply the tokenization function to the datasets
        print("Applying tokenization (this may take a while)...")
        # Use batched=True for speed, remove original text column
        train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
        val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
        print("Tokenization complete.")

        # Convert dataset into PyTorch-ready format
        train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])
        val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])
        print("Dataset format set to PyTorch tensors.")

        # Save the tokenized datasets so we don't have to redo this step
        print(f"Saving tokenized datasets to:")
        print(f"  Train: {TOKENIZED_TRAIN_PATH}")
        print(f"  Val:   {TOKENIZED_VAL_PATH}")
        train_dataset.save_to_disk(TOKENIZED_TRAIN_PATH)
        val_dataset.save_to_disk(TOKENIZED_VAL_PATH)
        print("\nTokenized datasets saved successfully.")

        # Show a preview of one sample’s tokenized input
        print("\nSample tokenized train entry (first 20 input_ids):")
        print(train_dataset[0]['input_ids'][:20])

    except Exception as e:
        print(f"An error occurred during tokenization: {e}")

In [ ]:
# Training Setup
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, AutoTokenizer
from datasets import load_from_disk
import torch
import os

# Flags to track success
datasets_loaded = False
can_train = False

# --- Load Tokenized Datasets ---
print("\nLoading tokenized datasets from disk...")
if os.path.exists(TOKENIZED_TRAIN_PATH) and os.path.exists(TOKENIZED_VAL_PATH):
    try:
        train_dataset = load_from_disk(TOKENIZED_TRAIN_PATH)
        val_dataset = load_from_disk(TOKENIZED_VAL_PATH)
        print(f"Loaded {len(train_dataset)} train and {len(val_dataset)} validation samples.")

        # If in debug mode, reduce dataset size for quick testing
        if DEBUG_MAX_TRAIN_SAMPLES and DEBUG_MAX_TRAIN_SAMPLES > 0:
            if len(train_dataset) > DEBUG_MAX_TRAIN_SAMPLES:
                 train_dataset = train_dataset.shuffle(seed=RANDOM_STATE).select(range(DEBUG_MAX_TRAIN_SAMPLES))
                 print(f"Using subset of {len(train_dataset)} training samples for this run.")
            else:
                 print(f"Debug sample size ({DEBUG_MAX_TRAIN_SAMPLES}) >= dataset size ({len(train_dataset)}). Using full loaded training set.")

        datasets_loaded = True
    except Exception as e:
        print(f"Error loading datasets from disk: {e}")
        datasets_loaded = False
else:
    print("Error: Tokenized dataset directories not found. Cannot proceed.")
    datasets_loaded = False

# Continue only if datasets loaded successfully
if datasets_loaded:
    # --- Device Setup ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # --- Load Pretrained Model and Tokenizer ---
    print(f"Loading base model: {BASE_MODEL_NAME}")
    try:
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME)
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

        # Make sure pad token is set (especially for GPT-2 style models)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            if model.config.pad_token_id is None:
                model.config.pad_token_id = tokenizer.eos_token_id
            print("Set pad_token for loaded tokenizer and potentially model config.")

        model.to(device)
        print(f"Model loaded to {device}.")


        # --- Data Collator ---
        # Prepares batches for causal language modeling
        data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
        print("Data collator initialized.")

        # --- Training Arguments ---

        print("Defining Training Arguments...")
        training_args = TrainingArguments(
            output_dir=MODEL_OUTPUT_DIR,
            overwrite_output_dir=True,
            num_train_epochs=NUM_TRAIN_EPOCHS,
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=TRAIN_BATCH_SIZE_PER_DEVICE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            weight_decay=WEIGHT_DECAY,


            save_strategy=SAVE_STRATEGY,
            save_total_limit=SAVE_TOTAL_LIMIT,
            load_best_model_at_end=False,

            logging_dir=LOGGING_DIR,
            logging_strategy="steps",
            logging_steps=LOGGING_STEPS,
            logging_first_step=True,

            fp16=FP16_TRAINING,

            report_to="none",
            disable_tqdm=False,
            save_safetensors=True,
            seed=RANDOM_STATE,
        )
        print("Training Arguments defined.")

        # --- Initialize Trainer ---
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=tokenizer,
            data_collator=data_collator,
        )
        print("Trainer initialized successfully.")
        can_train = True

    except Exception as e:
        print(f"An error occurred during model loading or Trainer setup: {e}")
        can_train = False
else:
    print("Skipping Trainer setup because datasets failed to load.")
    can_train = False

In [ ]:
if can_train:
    print("\n🚀 Starting Fine-Tuning... 🚀")
    try:
        # Start training
        train_result = trainer.train()

        # Save the final model (trainer already saves best model if configured)
        print(f"\nSaving final model (best checkpoint) to: {FINAL_MODEL_PATH}")
        trainer.save_model(FINAL_MODEL_PATH) # Saves model and tokenizer

        # Log metrics and calculate perplexity
        metrics = train_result.metrics
        trainer.log_metrics("train", metrics)
        trainer.save_metrics("train", metrics)

        if "train_loss" in metrics:
            train_perplexity = math.exp(metrics["train_loss"])
            print(f"Final Training Loss: {metrics['train_loss']:.4f} | Perplexity: {train_perplexity:.4f}")
        else:
             print("Training loss not found in metrics.")

        # --- Evaluate Model ---
        print("\nEvaluating final model on validation set...")
        eval_metrics = trainer.evaluate()
        trainer.log_metrics("eval", eval_metrics)
        trainer.save_metrics("eval", eval_metrics)

        if "eval_loss" in eval_metrics:
            eval_perplexity = math.exp(eval_metrics["eval_loss"])
            print(f"Final Evaluation Loss: {eval_metrics['eval_loss']:.4f} | Perplexity: {eval_perplexity:.4f}")
        else:
             print("Evaluation loss not found in metrics.")

        print(f"\n✅ Training complete! Final model saved at: {FINAL_MODEL_PATH}")

    except Exception as e:
        print(f"\n❌ An error occurred during training: {e}")
else:
    print("\nSkipping training because setup failed.")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# --- Load Fine-Tuned Model and Tokenizer ---
print(f"\n--- Testing Generation from Fine-Tuned Model ---")
if os.path.exists(FINAL_MODEL_PATH):
    print(f"Loading fine-tuned model from: {FINAL_MODEL_PATH}")
    try:
        # Load the tokenizer and model saved by Trainer
        ft_tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_PATH)
        ft_model = AutoModelForCausalLM.from_pretrained(FINAL_MODEL_PATH)

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        ft_model.to(device)
        ft_model.eval()
        print(f"Fine-tuned model loaded successfully to {device}.")
        model_loaded_for_test = True
    except Exception as e:
        print(f"Error loading fine-tuned model/tokenizer: {e}")
        model_loaded_for_test = False
else:
    print(f"Error: Fine-tuned model directory not found at {FINAL_MODEL_PATH}")
    model_loaded_for_test = False

# --- Generate Text Using Pipeline ---
if model_loaded_for_test:
    prompt = "ሰላም"  # Your test prompt here
    print(f"\nGenerating text with prompt: '{prompt}'")

    print(f"Using pipeline with num_beams={TEST_NUM_BEAMS}...")
    try:
        text_generator = pipeline(
            'text-generation',
            model=ft_model,
            tokenizer=ft_tokenizer,
            device=ft_model.device.index if device.type == 'cuda' else -1
        )
        outputs = text_generator(
            prompt,
            max_new_tokens=TEST_MAX_NEW_TOKENS,
            num_beams=TEST_NUM_BEAMS,
            no_repeat_ngram_size=TEST_NO_REPEAT_NGRAM_SIZE,

            pad_token_id=ft_tokenizer.eos_token_id
        )
        generated_text = outputs[0]['generated_text']
        print("\n📝 Generated Text (Pipeline):\n", generated_text)
    except Exception as e:
        print(f"Error during pipeline generation: {e}")
else:
    print("Skipping generation test as model loading failed.")

print("--- End Generation Test ---")

In [ ]:
# === Chatbot Cell ===
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import os

print(f"\n--- Initializing Chatbot ---")
if os.path.isdir(FINAL_MODEL_PATH):
    print(f"Loading model for chatbot from: {FINAL_MODEL_PATH}")
    try:
        chat_tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_PATH)
        chat_model = AutoModelForCausalLM.from_pretrained(FINAL_MODEL_PATH)

        if chat_tokenizer.pad_token is None:
            chat_tokenizer.pad_token = chat_tokenizer.eos_token
        if chat_model.config.pad_token_id is None:
            chat_model.config.pad_token_id = chat_tokenizer.eos_token_id

        chat_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        chat_model.to(chat_device)
        chat_model.eval()
        print(f"Chatbot model ready on device: {chat_device}")
        chatbot_ready = True
    except Exception as e:
        print(f"Error loading model/tokenizer for chat: {e}")
        chatbot_ready = False
else:
    print(f"Error: Model directory not found at {FINAL_MODEL_PATH}. Cannot start chat.")
    chatbot_ready = False


def generate_chat_response(prompt, model, tokenizer, device):
    """Generates a chatbot response using sampling."""
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

    try:
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=GENERATION_MAX_NEW_TOKENS,
                temperature=GENERATION_TEMPERATURE,
                top_k=GENERATION_TOP_K,
                top_p=GENERATION_TOP_P,
                repetition_penalty=GENERATION_REPETITION_PENALTY,
                no_repeat_ngram_size=GENERATION_NO_REPEAT_NGRAM_SIZE,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        response = full_text
        if prompt and full_text.startswith(prompt):
            response = full_text[len(prompt):].strip()

        response = response.split('\n')[0]
        response = response.replace("...", "").strip()

        if not response and full_text != prompt:
            return "ይቅርታ፣ ምላሽ ማመንጨት አልቻልኩም።"
        elif not response:
            return "ይቅርታ፣ ምላሽ ማመንጨት አልቻልኩም።"

        return response

    except Exception as e:
        print(f"\nError during text generation: {e}")
        return "ይቅርታ፣ በምላሹ ወቅት ስህተት አጋጥሟል።"


def run_chatbot():
    """Starts and manages the interactive chat session."""
    if not chatbot_ready:
        print("Chatbot cannot start because the model failed to load.")
        return

    print("\n--- የአማርኛ የውይይት ቦት ---")
    print("ሰላም! እንዴት ልረዳዎት እችላለሁ?")
    print("ለመውጣት 'ውጣ' ወይም 'exit' ብለው ይጻፉ።")
    print("-" * 30)

    while True:
        try:
            user_input = input("እርስዎ: ")
            if user_input.lower().strip() in ["ውጣ", "exit", "quit", "ቻው"]:
                print("ቦት: ደህና ሁኑ!")
                break

            if not user_input.strip():
                print("ቦት: እባክዎ ጥያቄዎን ያስገቡ።")
                continue

            bot_response = generate_chat_response(user_input, chat_model, chat_tokenizer, chat_device)
            print(f"ቦት: {bot_response}")

        except KeyboardInterrupt:
            print("\nቦት: ውይይቱ ተቋርጧል። ደህና ሁኑ!")
            break
        except EOFError:
            print("\nቦት: የግቤት ስህተት ተፈጥሯል። ደህና ሁኑ!")
            break
        except Exception as e:
            print(f"\nAn unexpected error occurred in the chat loop: {e}")
            print("ቦት: ይቅርታ ያልተጠበቀ ስህተት አጋጥሟል። እባክዎ እንደገና ይሞክሩ።")

run_chatbot()
